# Cuadernillo de práctica (autograding con nbgrader)

Este cuadernillo tiene dos ejercicios calificados con **nbgrader**. Cada ejercicio tiene:

1. Una celda donde escribes tu solución (marcada con `### BEGIN SOLUTION` / `### END SOLUTION`, que es la convención que usa nbgrader para generar la versión que reciben los estudiantes).
2. Una celda de verificación que corre pruebas automáticas y **captura cualquier error** en vez de detener el cuadernillo.

Al final se arma el `JSON` que se enviaría a un backend (que todavía no existe, así que por ahora solo se muestra) y se calculan las estadísticas del intento.

> Para usar esto dentro de un curso real de nbgrader necesitas `pip install nbgrader` y este archivo como la versión *source* del assignment; nbgrader se encarga de generar la versión que reciben los estudiantes (quitando el contenido entre `BEGIN SOLUTION` / `END SOLUTION`) y de correr las celdas marcadas como `grade` para calificar.

**Tal como está, los dos ejercicios empiezan sin terminar** (simulando a un estudiante a medias), para que al correr todo el cuadernillo veas primero cómo se capturan los errores. Después descomenta la solución dentro de `BEGIN SOLUTION` / `END SOLUTION` (o escribe la tuya) y borra el `raise NotImplementedError`, y vuelve a correr las celdas de verificación, el JSON y las estadísticas para ver cómo cambian.

In [2]:
import json
import traceback
from datetime import datetime, timezone

# Guarda el resultado de cada ejercicio verificado en esta sesión
resultados_ejercicios = []


def verificar_ejercicio(id_ejercicio, funcion_prueba, puntos_maximos=1):
    """
    Ejecuta funcion_prueba (que contiene los asserts del ejercicio).
    Si falla, captura el error en vez de detener la ejecución del cuadernillo,
    y guarda el resultado (aprobado/reprobado, puntos, error) en resultados_ejercicios.
    """
    resultado = {
        "ejercicio": id_ejercicio,
        "puntos_maximos": puntos_maximos,
        "puntos_obtenidos": 0,
        "aprobado": False,
        "error": None,
    }
    try:
        funcion_prueba()
        resultado["aprobado"] = True
        resultado["puntos_obtenidos"] = puntos_maximos
        print(f"[OK] {id_ejercicio}: correcto ({puntos_maximos} pts)")
    except AssertionError as e:
        resultado["error"] = {"tipo": "AssertionError", "mensaje": str(e) or "El resultado no es el esperado"}
        print(f"[FALLO] {id_ejercicio}: el resultado no es el esperado")
    except Exception as e:
        resultado["error"] = {
            "tipo": type(e).__name__,
            "mensaje": str(e),
            "traceback": traceback.format_exc(limit=2),
        }
        print(f"[ERROR] {id_ejercicio}: tu código lanzó un error ({type(e).__name__})")

    global resultados_ejercicios
    resultados_ejercicios = [r for r in resultados_ejercicios if r["ejercicio"] != id_ejercicio]
    resultados_ejercicios.append(resultado)
    return resultado

## Ejercicio 1 — `es_par`

Completa la función `es_par(n)` para que retorne `True` si `n` es par y `False` si es impar.

In [5]:
def es_par(n):
    return n % 2 == 0
    raise NotImplementedError("Elimina esta línea e implementa la función")

In [6]:
def _test_es_par():
    assert es_par(4) == True, "es_par(4) debería ser True"
    assert es_par(7) == False, "es_par(7) debería ser False"
    assert es_par(0) == True, "es_par(0) debería ser True"


verificar_ejercicio("ejercicio_1_es_par", _test_es_par, puntos_maximos=2)

[OK] ejercicio_1_es_par: correcto (2 pts)


{'ejercicio': 'ejercicio_1_es_par',
 'puntos_maximos': 2,
 'puntos_obtenidos': 2,
 'aprobado': True,
 'error': None}

## Ejercicio 2 — `suma_lista`

Completa la función `suma_lista(lista)` para que retorne la suma de todos los números de la lista.

In [9]:
def suma_lista(lista):

    return sum(lista)
    
    raise NotImplementedError("Elimina esta línea e implementa la función")

In [10]:
def _test_suma_lista():
    assert suma_lista([1, 2, 3]) == 6
    assert suma_lista([]) == 0
    assert suma_lista([-1, 1]) == 0


verificar_ejercicio("ejercicio_2_suma_lista", _test_suma_lista, puntos_maximos=3)

[OK] ejercicio_2_suma_lista: correcto (3 pts)


{'ejercicio': 'ejercicio_2_suma_lista',
 'puntos_maximos': 3,
 'puntos_obtenidos': 3,
 'aprobado': True,
 'error': None}

## Envío al backend (todavía no existe)

Dejamos la configuración y la función de envío listas, pero desactivadas. Cuando tengas el backend, solo cambias `ENVIAR_AL_BACKEND` a `True`.

In [11]:
# --- Configuración del backend ---
BACKEND_URL = "https://tu-backend.example.com/api/resultados"
ENVIAR_AL_BACKEND = False  # cambia a True cuando el backend esté disponible

# Si este cuadernillo corre dentro de tu JupyterHub con LTI, aquí puedes
# tomar el id real desde las variables de entorno que vimos antes, por ejemplo:
# import os
# ID_ESTUDIANTE = os.environ.get("ALUMNO_EMAIL", "desconocido")
ID_ESTUDIANTE = "por-definir"


def construir_payload():
    return {
        "estudiante_id": ID_ESTUDIANTE,
        "cuaderno": "cuadernillo_ejemplo",
        "fecha": datetime.now(timezone.utc).isoformat(),
        "ejercicios": resultados_ejercicios,
        "puntaje_total": sum(r["puntos_obtenidos"] for r in resultados_ejercicios),
        "puntaje_maximo": sum(r["puntos_maximos"] for r in resultados_ejercicios),
    }


def enviar_resultados(payload):
    if not ENVIAR_AL_BACKEND:
        print("Envío desactivado (ENVIAR_AL_BACKEND = False). Solo se muestra el JSON abajo.")
        return None
    import requests
    respuesta = requests.post(BACKEND_URL, json=payload, timeout=5)
    respuesta.raise_for_status()
    return respuesta.json()

## JSON que se enviaría al backend

In [12]:
payload = construir_payload()
print(json.dumps(payload, indent=2, ensure_ascii=False))

# Cuando el backend exista, solo necesitas:
# enviar_resultados(payload)

{
  "estudiante_id": "por-definir",
  "cuaderno": "cuadernillo_ejemplo",
  "fecha": "2026-07-19T02:22:39.563376+00:00",
  "ejercicios": [
    {
      "ejercicio": "ejercicio_1_es_par",
      "puntos_maximos": 2,
      "puntos_obtenidos": 2,
      "aprobado": true,
      "error": null
    },
    {
      "ejercicio": "ejercicio_2_suma_lista",
      "puntos_maximos": 3,
      "puntos_obtenidos": 3,
      "aprobado": true,
      "error": null
    }
  ],
  "puntaje_total": 5,
  "puntaje_maximo": 5
}


## Estadísticas del cuadernillo

In [13]:
def mostrar_estadisticas():
    total = len(resultados_ejercicios)
    aprobados = sum(1 for r in resultados_ejercicios if r["aprobado"])
    puntos_obtenidos = sum(r["puntos_obtenidos"] for r in resultados_ejercicios)
    puntos_maximos = sum(r["puntos_maximos"] for r in resultados_ejercicios)
    porcentaje = (puntos_obtenidos / puntos_maximos * 100) if puntos_maximos else 0

    print(f"Ejercicios verificados: {total}")
    print(f"Ejercicios aprobados:  {aprobados}/{total}")
    print(f"Puntaje: {puntos_obtenidos}/{puntos_maximos} ({porcentaje:.1f}%)")

    if total > 0 and aprobados == total:
        print("\nCompletaste todos los ejercicios del cuadernillo.")
    else:
        pendientes = [r["ejercicio"] for r in resultados_ejercicios if not r["aprobado"]]
        print(f"\nTe faltan por resolver: {', '.join(pendientes)}")


mostrar_estadisticas()

Ejercicios verificados: 2
Ejercicios aprobados:  2/2
Puntaje: 5/5 (100.0%)

Completaste todos los ejercicios del cuadernillo.
